# BakerySense
## AI Production Copilot for Independent Bakeries

**Kaggle Gemma 4 Good Hackathon — Submission**

*Authors:* Soh Wei Meng (`whymelabs`)  
*Repository:* https://github.com/wms2537/gemma-4-hack  
*Live demo:* https://bakerysense-web.swmengappdev.workers.dev (credentials: `demo@bakerysense.app` / `Demo2026DemoDemo` / tenant `favorita`)  
*Demo video:* see GitHub repo `docs/demo/demo-final.mp4` (1:52, 10 MB)

---

## 1. The problem

Independent bakeries throw out 30–40% of perishable production daily. In France alone, boulangers discard millions of unsold baguettes and croissants every week — the waste isn't ignorance, it's uncertainty. A baker can't know on Monday morning whether Tuesday will bring a school-holiday crowd or a rainstorm.

POS systems record what sold; they don't predict what to make. Spreadsheet templates require statistical literacy. Enterprise demand-planning software assumes a logistics team and a six-figure budget. A single-location bakery with five employees and a 4am start time has none of those.

## 2. The solution

BakerySense reduces the baker's decision to one photo and one tap. The dashboard shows a bake plan derived from a quantile forecast and a newsvendor decision under the bakery's specific cost-of-waste vs cost-of-stockout ratio. At 5pm, the baker photographs the display case — Gemma 4 counts remaining units from the image and returns markdown suggestions to clear inventory before close.

The design principle: **numeric work is deterministic; semantic work is LLM.** LightGBM produces forecast numbers, an approximate-SHAP walker produces feature attributions, the newsvendor equation produces production quantity. Gemma 4 reads those outputs and writes the explanation. It does not produce numbers.

## 3. Gemma 4's role

Gemma 4 E4B is the merchant-facing layer. Three jobs:

1. **Multimodal ingestion.** Display-case photo → Gemma 4 → structured JSON count whitelisted against known SKUs.
2. **Tool routing.** Free-form merchant questions emit OpenAI-compatible tool calls (5 registered functions: `forecast_point`, `explain_drivers`, `waste_risk`, `list_skus`, `close_out_day`).
3. **Explanations.** SHAP driver arrays → plain-language sentences ("Last Tuesday's sales of 141 baguettes are pulling the forecast up").

Why Gemma 4: Apache 2.0 license (no royalties), E4B fits in 16 GB RAM in GGUF Q4_K_M (matches MacBook Pros most independents already own — no API cost, no data egress), handles French bakery vocabulary natively. Eight LLM connector presets let operators run locally or in cloud with one click.

---

## 4. Architecture

**Cloudflare stack.** Next.js 16 via OpenNext on Cloudflare Workers; D1 (SQLite) transactional, KV for hot config + model pointers, R2 for model artifacts + SHAP feature store, Queues for async chat + retrain. Cold-start <50 ms.

**Pure-TypeScript LightGBM inference.** Trained model exported as JSON (human-auditable, no binary serialisation), walked by a pure-TS tree walker (`gbm-walker.ts`) inside the Worker. No Python at request time. JS↔Python parity 700/700 within 1×10⁻⁴ tolerance.

**V1.5 forecasting layer (the production architecture):**

1. **Population prior** at `(family × dow)` — anchors the median where retail seasonality is strong.
2. **Per-quantile maturity blend** (Tier 4) — at maturity the median stays with the prior (lower WAPE), tails switch to LightGBM (calibrated for newsvendor).
3. **TimesFM-2 tail** (Tier 6) — when `TIMESFM_ENDPOINT` is set, q0.8/q0.9 route to a foundation model with measurably better calibration.

All three live in production today. The `forecaster` field in the API response reports which path served each request: `population_prior_v1` (cold) → `perq_blend_v1` (Tier 4) → `perq_blend_v2` (Tier 6 with TimesFM tail).

## 5. Results

On the **French Bakery** Kaggle dataset (`matthieugimbert/french-bakery-daily-sales`, 28-day × 20-SKU holdout, identical per-SKU fit and horizon for every method):

| Forecaster | WAPE | MASE | pinball-q0.5 | pinball-q0.9 |
|---|---|---|---|---|
| Seasonal-naive (lag-7)            | 0.341 | 1.000 | 3.27 | – |
| AutoARIMA (`statsforecast`)         | 0.548 | 1.610 | 5.26 | – |
| AutoETS (`statsforecast`)           | 0.271 | 0.796 | 2.60 | – |
| CrostonClassic (intermittent)     | 0.764 | 2.244 | 7.34 | – |
| V1 LightGBM (with weather + lag-365) | 0.245 | 0.719 | 2.35 | **1.15** |
| V1.5 population prior (cold-start) | 0.212 | 0.623 | 2.04 | – |
| **V1.5 PER-QUANTILE blend (Tier 4 — production)** | **0.212** | **0.623** | **2.04** | **1.15** |
| TimesFM-2.0-500m zero-shot               | 0.314 | 0.921 | 3.01 | – |
| **V1.5 PRIOR + TimesFM TAIL (Tier 6 — live)** | **0.212** | **0.623** | **2.04** | **1.09** |

**The V1.5 production forecaster posts WAPE 22% below the best published baseline (AutoETS 0.271).**

The mechanism: per-quantile alpha. The `(family × dow)` prior wins the median (lower WAPE — ignores recent-shock noise). LightGBM wins q0.9 (calibrated for newsvendor). Per-quantile blending keeps both, smoothly ramped by `maturity = clip(actuals_count / 90, 0, 1)` so cold tenants still see pure prior.

## 6. Cross-dataset generalization (9 published benchmarks)

We extended the architecture beyond French Bakery to verify it isn't dataset-specific. Same forecasters, six published competition benchmarks plus three GIFT-Eval datasets:

| Dataset | Ours | Best published / our rank |
|---|---|---|
| French Bakery (retail + weather) | **WAPE 0.212** | AutoETS 0.271 — V1.5 wins by 22% |
| NN5 Daily | WAPE 0.197 | AutoETS 0.192 (narrow loss) |
| **M4 Daily** (4,227 series) | **sMAPE 2.16** | M4 winner ES-RNN 3.05 — beats every published method |
| **Kaggle Web Traffic** (1,095 teams) | **SMAPE 38.83** | top 50 of 1,095 teams (top 5%) |
| M5 Accuracy (5,558 teams) | **WRMSSE 0.713** | beats naive 0.91 by 22%; mid-pack of field |
| **M5 Uncertainty** (909 teams) | **WSPL 0.138** | top-tier validation range (winner 0.157 private) |
| M4 Monthly (5K subset) | sMAPE 10.20 | better than Chronos-Large 12.71 (caveat: subsample) |
| Tourism Monthly | sMAPE 20.79 | worse than Chronos ~18.0 (small-N monthly) |
| Hospital | MASE 0.876 | worse than Chronos / MOIRAI ~0.75 |

**Wins on retail-daily** where pre-training fits (M5, Kaggle Web Traffic, M4 daily). **Loses on small-N monthly + weekly counts** where Chronos / MOIRAI were specifically tuned. The transferable claim is the *architectural pattern*, not any particular foundation model.

## 7. The architecture-vs-model finding

**Tier 23, the cleanest experiment of the project:** swap TimesFM-2 for Amazon's Chronos-Bolt under the *exact same* pipeline (per-quantile + per-level routing + tail extrapolation), score on M5 Uncertainty.

| Foundation model | WSPL |
|---|---|
| TimesFM-2.0-500m + Tier 21 architecture | **0.1379** |
| Chronos-Bolt-Base + Tier 21 architecture | **0.1396** |
| Delta | 1.2% |

Both well below the M5 Uncertainty winner's 0.157 private score. **The wiring transfers; the foundation model is fungible at the 1% level.** This is the most production-defensible claim because it's *forward-compatible* with whatever 2027 foundation model comes next.

## 8. The architectural recipe

After 23 tiers of experiments (full log: [`docs/research/tier-scorecard.md`](https://github.com/wms2537/gemma-4-hack/blob/master/docs/research/tier-scorecard.md)), the pattern that *generalizes*:

**For point forecasts on dense daily retail:**
1. Compute the population prior at `(family × dow)` from history.
2. Compute a foundation-model forecast at the right intermediate aggregation level (~70 series for M5).
3. Disaggregate top-down via static last-28-day revenue shares.
4. Per-quantile blend at maturity — prior at median, GBM/FM at q0.9.

**For uncertainty / quantile envelopes:**
1. Forecast at multiple aggregation levels (L9 + L10 + L11 for M5).
2. Linear-extrapolate the quantile tails beyond the FM's pre-trained range.
3. Route each evaluation level to its directly-targeted forecast.
4. Disaggregate via static within-group leaf shares; enforce monotonicity.

**Compute footprint:** 71 + 3,049 + 9,147 = 12,266 forecast calls for M5 Uncertainty Tier 21 — ~10 minutes on Apple Silicon GPU. Vs the M5 winner's 12-model ensemble that took weeks of training.

## 9. Tracks and deployment

**Main + Impact:** food waste (SDG 12.3) and small-merchant economic resilience; offline-first design works in regions with unreliable connectivity.

**Unsloth:** Gemma 4 E4B without fine-tuning; QLoRA on bakery vocabulary is a documented stretch goal.

**Ollama:** the `ollama-tunnel` connector points at `http://localhost:11434` for fully on-device inference. One of eight first-class connectors.

**llama.cpp:** the Python demo layer uses llama-cpp-python with `BAKERYSENSE_MODEL_REPO` / `_FILE` env vars pointing to the GGUF artifact.

**Test matrix:** 185 tests green — 49 Python (forecaster, newsvendor, SHAP, eval), 167 Cloudflare Workers (API routes, auth, agent loop, feedback loop, V2 architecture), 11 unit (JS walker, forecast router, conformal, hierarchical, context compactor), 7 Playwright E2E.

**License:** CC-BY-4.0 per Rules §2.5.

## 10. What's next

Out of scope for this submission, on the roadmap:

- **TimesFM-2 cold-start sidecar** — router stub already shipped; needs Cloudflare Container backend
- **Tenant QLoRA** fine-tunes per-bakery on chat corrections
- **POS integrations** (Square, Lightspeed, SumUp) for actuals capture
- **Cloudflare Container retrain** so the actuals → retrain → publish → hot-swap loop runs without local tooling

---

## Repository structure

Code, demo video, and full 23-tier research log: **https://github.com/wms2537/gemma-4-hack**

Key files:
- `bakerysense-web/` — production Next.js + Cloudflare Workers app
- `src/bakerysense/` — Python forecaster + retrain pipeline
- `docs/research/tier-scorecard.md` — 23-tier experimental log (positive + negative)
- `docs/architecture/v2-migration.md` — Sprint 0–5 architecture map
- `docs/demo/demo-final.mp4` — walkthrough video (1:52)
- `docs/demo/writeup.md` — full submission writeup (1,479 words)

**Live demo:** https://bakerysense-web.swmengappdev.workers.dev

In [ ]:
# This notebook is documentation-only — the actual code lives in the GitHub repo above.
# To reproduce the headline benchmark numbers, clone the repo and run:
#
#   pip install -e '.[dev]'
#   python scripts/benchmark_vs_baselines.py     # Tier 1-9 on French Bakery
#   python scripts/benchmark_m5_uncertainty_v4.py # Tier 21 — top-tier WSPL on M5
#   python scripts/benchmark_m5_chronos.py        # Tier 23 — architecture-vs-model proof
#
# Each script prints the numbers shown in the tables above.
print('See https://github.com/wms2537/gemma-4-hack for code, full research log, and live demo URL.')